# Hardware – Addition avec DMA

In [2]:
from pynq import Overlay, allocate, PL
import numpy as np

In [3]:
PL.reset()
overlay=Overlay("design_1.bit")
print("IP disponibles :")
for name in overlay.ip_dict:
    print(" -",name)

IP disponibles :
 - adder_dma_0
 - axi_dma_0
 - processing_system7_0


In [4]:
dma=None; dma_name=None
for name in overlay.ip_dict:
    obj=getattr(overlay,name)
    if hasattr(obj,"sendchannel") and hasattr(obj,"recvchannel"):
        dma=obj; dma_name=name; break
if dma is None:
    raise RuntimeError("Aucun DMA MM2S+S2MM trouvé")

adder=None; adder_name=None
for name in overlay.ip_dict:
    if "adder" in name.lower():
        adder=getattr(overlay,name); adder_name=name; break
if adder is None:
    raise RuntimeError("Aucune IP adder trouvée")

print("DMA :",dma_name)
print("IP  :",adder_name)

DMA : axi_dma_0
IP  : adder_dma_0


In [5]:
registers=overlay.ip_dict[adder_name].get("registers",{})
print(registers)
SIZE_OFFSET=registers["size"]["address_offset"]
CTRL_OFFSET=0x00

{'CTRL': {'address_offset': 0, 'size': 32, 'access': 'read-write', 'description': 'Control signals', 'fields': {'AP_START': {'bit_offset': 0, 'bit_width': 1, 'access': 'read-write', 'description': "Control signal Register for 'ap_start'."}, 'AP_DONE': {'bit_offset': 1, 'bit_width': 1, 'access': 'read-only', 'description': "Control signal Register for 'ap_done'."}, 'AP_IDLE': {'bit_offset': 2, 'bit_width': 1, 'access': 'read-only', 'description': "Control signal Register for 'ap_idle'."}, 'AP_READY': {'bit_offset': 3, 'bit_width': 1, 'access': 'read-only', 'description': "Control signal Register for 'ap_ready'."}, 'RESERVED_1': {'bit_offset': 4, 'bit_width': 3, 'access': 'read-only', 'description': 'Reserved.  0s on read.'}, 'AUTO_RESTART': {'bit_offset': 7, 'bit_width': 1, 'access': 'read-write', 'description': "Control signal Register for 'auto_restart'."}, 'RESERVED_2': {'bit_offset': 8, 'bit_width': 1, 'access': 'read-only', 'description': 'Reserved.  0s on read.'}, 'INTERRUPT': {'b

In [6]:
def add_vectors_fpga(A,B):
    A=np.asarray(A,dtype=np.int32)
    B=np.asarray(B,dtype=np.int32)
    if A.shape!=B.shape:
        raise ValueError("A et B doivent avoir la même taille")
    N=len(A)
    ib=allocate(shape=(2*N,),dtype=np.int32)
    ob=allocate(shape=(N,),dtype=np.int32)
    try:
        ib[0::2]=A; ib[1::2]=B
        ib.flush(); ob.flush()
        adder.write(SIZE_OFFSET,N)
        dma.recvchannel.transfer(ob)
        adder.write(CTRL_OFFSET,1)
        dma.sendchannel.transfer(ib)
        dma.sendchannel.wait(); dma.recvchannel.wait()
        ob.invalidate()
        return np.array(ob,dtype=np.int32)
    finally:
        ib.freebuffer(); ob.freebuffer()

In [9]:
A=np.array([1,2,3,4,5],dtype=np.int32)
B=np.array([10,20,30,40,50],dtype=np.int32)
hw=add_vectors_fpga(A,B)
print("A        =",A)
print("B        =",B)
print("FPGA     =",hw)
print("Correct  =",np.array_equal(hw,sw))

A        = [1 2 3 4 5]
B        = [10 20 30 40 50]
FPGA     = [11 22 33 44 55]
Correct  = False


In [10]:
rng=np.random.default_rng(seed=42)
N=1024
A=rng.integers(-100,101,size=N,dtype=np.int32)
B=rng.integers(-100,101,size=N,dtype=np.int32)
hw=add_vectors_fpga(A,B)
print("Nombre d'additions :",N)
print("Résultats corrects :",np.array_equal(hw,sw))
print("FPGA     :",hw[:10])

Nombre d'additions : 1024
Résultats corrects : True
FPGA     : [ -18  -34  -54  -56    0   39  -86  -26  -14 -119]
